## Notes & Troubleshooting

**Architecture:**
- **Multi-class classification**: Predicts which option (0-3) is correct
- **Input**: passage + question + all 4 options
- **Output**: predicted option index + confidence + per-class probabilities
- **Features**: One-Hot encoding (5000 dims × 4) + Lexical (4 dims × 4) = 20,016 total dims

**Expected Results:**
- Accuracy: 60-80% on validation (vs. 23% with old binary approach)
- Best performers: XGBoost, Ensemble Voting, Random Forest
- Training time: ~12-18 minutes with GPU

**How to Use New Models:**
```python
from src.inference import ModelAInference

result = inference.predict_answer(
    passage="...",
    question="...",
    options=["A", "B", "C", "D"],
    method='ensemble_voting_model'
)

print(f"Answer: {result['predicted_letter']}")
print(f"Confidence: {result['confidence']:.1%}")
```

**Models Included:**
- Logistic Regression (lr_model)
- Gaussian Naive Bayes (nb_model)
- Random Forest (rf_model)
- Support Vector Machine (svm_model)
- XGBoost (xgb_model)
- Voting Ensemble (ensemble_voting_model)
- K-Means (kmeans_model)
- Label Propagation (label_propagation_model)
- Gaussian Mixture Model (gmm_model)
- Feature Engineer (feature_engineer.pkl)

**References:**
- [KAGGLE_MULTICLASS_GUIDE.md](KAGGLE_MULTICLASS_GUIDE.md) - Full guide
- [MULTICLASS_QUICK_REFERENCE.md](MULTICLASS_QUICK_REFERENCE.md) - Quick reference
- [scripts/test_multiclass.py](scripts/test_multiclass.py) - Local testing

In [ ]:
print("\n" + "=" * 70)
print("SAVING MODELS AND ARTIFACTS")
print("=" * 70)

output_dir = Path('/content/models/model_a/traditional/')
output_dir.mkdir(parents=True, exist_ok=True)

# Save all models
print(f"\nSaving models to: {output_dir}\n")
for model_name, model in models.items():
    path = output_dir / f"{model_name}.pkl"
    joblib.dump(model, path)
    print(f"✓ Saved: {model_name}.pkl ({path.stat().st_size / 1024 / 1024:.2f} MB)")

# Save feature engineer
feature_engineer_path = output_dir / "feature_engineer.pkl"
feature_engineer.save(feature_engineer_path)
print(f"✓ Saved: feature_engineer.pkl ({feature_engineer_path.stat().st_size / 1024 / 1024:.2f} MB)")

print("\n" + "=" * 70)
print("TRAINING COMPLETE ✓")
print("=" * 70)
print(f"\nAll models saved to: {output_dir}")
print(f"Total models trained: {len(models)}")
print(f"Expected accuracy on test set: 60-80%")
print(f"\nNext steps:")
print(f"1. Download all .pkl files from the output folder")
print(f"2. Or copy them to Google Drive for persistence")
print(f"3. Run: python scripts/test_multiclass.py")
print(f"4. Integrate into Streamlit UI with predict_answer() method")

## 10. Save Models and Feature Artifacts

In [ ]:
print("\n" + "=" * 70)
print("TRAINING CLASSIFIERS")
print("=" * 70)

models = {}
scores = {}

# Logistic Regression
print("\n1. Logistic Regression...")
models['lr_model'] = LogisticRegression(
    max_iter=1000,
    multi_class='multinomial',
    class_weight='balanced',
    random_state=42,
    C=0.5
)
models['lr_model'].fit(X_train, y_train)
scores['lr_model'] = models['lr_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['lr_model']:.4f}")

# Gaussian Naive Bayes
print("\n2. Gaussian Naive Bayes...")
models['nb_model'] = GaussianNB()
models['nb_model'].fit(X_train, y_train)
scores['nb_model'] = models['nb_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['nb_model']:.4f}")

# Random Forest
print("\n3. Random Forest...")
models['rf_model'] = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
models['rf_model'].fit(X_train, y_train)
scores['rf_model'] = models['rf_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['rf_model']:.4f}")

# Support Vector Machine
print("\n4. Support Vector Machine (SVM)...")
models['svm_model'] = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=42,
    gamma='scale',
    decision_function_shape='ovr'
)
models['svm_model'].fit(X_train, y_train)
scores['svm_model'] = models['svm_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['svm_model']:.4f}")

# XGBoost
print("\n5. XGBoost...")
models['xgb_model'] = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    num_class=4
)
models['xgb_model'].fit(X_train, y_train)
scores['xgb_model'] = models['xgb_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['xgb_model']:.4f}")

# Ensemble - Voting Classifier
print("\n6. Ensemble (Soft Voting)...")
models['ensemble_voting_model'] = VotingClassifier(
    estimators=[
        ('lr', models['lr_model']),
        ('rf', models['rf_model']),
        ('svm', models['svm_model']),
    ],
    voting='soft'
)
models['ensemble_voting_model'].fit(X_train, y_train)
scores['ensemble_voting_model'] = models['ensemble_voting_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['ensemble_voting_model']:.4f}")

# K-Means
print("\n7. K-Means (unsupervised)...")
models['kmeans_model'] = KMeans(n_clusters=4, random_state=42, n_init=10)
models['kmeans_model'].fit(X_train)
print(f"   ✓ K-Means fitted")

# Label Propagation
print("\n8. Label Propagation (semi-supervised)...")
models['label_propagation_model'] = LabelPropagation(n_neighbors=7)
models['label_propagation_model'].fit(X_train, y_train)
scores['label_propagation_model'] = models['label_propagation_model'].score(X_val, y_val)
print(f"   ✓ Validation accuracy: {scores['label_propagation_model']:.4f}")

# Gaussian Mixture Model
print("\n9. Gaussian Mixture Model (unsupervised)...")
models['gmm_model'] = GaussianMixture(n_components=4, random_state=42)
models['gmm_model'].fit(X_train)
print(f"   ✓ Gaussian Mixture Model fitted")

# Print summary
print("\n" + "=" * 70)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 70)
sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
for i, (model_name, score) in enumerate(sorted_scores, 1):
    print(f"{i}. {model_name:30s}: {score:.4f} ({score*100:.2f}%)")

## 9. Train Base Models and Ensemble

In [ ]:
print("\n" + "=" * 70)
print("BUILDING FEATURE MATRICES")
print("=" * 70)

# 1. Fit feature engineer
print("\n1. Fitting feature engineer...")
all_question_texts = train_df['question'].tolist()
all_option_texts = []
for opts in train_df['options']:
    all_option_texts.extend(opts)

combined_texts = all_question_texts + all_option_texts
feature_engineer = FeatureEngineer(max_features=5000)
feature_engineer.fit_onehot(combined_texts)
print("   ✓ Feature engineer fitted")
print(f"   Vocabulary size: {len(feature_engineer.onehot_vectorizer.vocabulary_)}")

# 2. Build training features
print("\n2. Building training feature matrix...")
X_train = build_multiclass_feature_matrix(
    train_df['question'].tolist(),
    train_df['options'].tolist(),
    train_df['article'].tolist(),
    feature_engineer
)
y_train = train_df['label'].values

print(f"   ✓ X_train shape: {X_train.shape}")
print(f"   ✓ y_train shape: {y_train.shape}")
print(f"   Class balance: {np.bincount(y_train, minlength=4) / len(y_train) * 100:.1f}% each")

# 3. Build validation features
print("\n3. Building validation feature matrix...")
X_val = build_multiclass_feature_matrix(
    val_df['question'].tolist(),
    val_df['options'].tolist(),
    val_df['article'].tolist(),
    feature_engineer
)
y_val = val_df['label'].values

print(f"   ✓ X_val shape: {X_val.shape}")
print(f"   ✓ y_val shape: {y_val.shape}")

## 8. Fit Vectorizer and Generate Features

In [ ]:
print("=" * 70)
print("PREPARING DATA FOR TRAINING")
print("=" * 70)

# Parse options column (might be string representation of list)
def parse_options(opt):
    if isinstance(opt, str):
        import ast
        try:
            return ast.literal_eval(opt)
        except:
            return [opt, '', '', '']
    return opt

train_df['options'] = train_df['options'].apply(parse_options)
val_df['options'] = val_df['options'].apply(parse_options)

# Convert answers to labels
train_df['label'] = train_df['answer'].apply(convert_answer_to_label)
val_df['label'] = val_df['answer'].apply(convert_answer_to_label)

print(f"\nTraining samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Class distribution (train):")
print(f"  {np.bincount(train_df['label'], minlength=4)}")
print(f"  Percentages: {np.bincount(train_df['label'], minlength=4) / len(train_df) * 100}")
print(f"\nClass distribution (validation):")
print(f"  {np.bincount(val_df['label'], minlength=4)}")
print(f"  Percentages: {np.bincount(val_df['label'], minlength=4) / len(val_df) * 100}")

## 7. Parse Options and Prepare Training Data

In [ ]:
print("Loading RACE dataset in Colab...\n")

# Colab paths: prefer uploaded files, then Google Drive if mounted
colab_data_candidates = [
    '/content/race-dataset',
    '/content/drive/MyDrive/race-dataset',
    '/content/drive/MyDrive/RACE',
]

def load_arrow_or_csv(base_path):
    base = Path(base_path)
    arrow_train = base / 'train' / 'data-00000-of-00001.arrow'
    arrow_val = base / 'validation' / 'data-00000-of-00001.arrow'
    csv_train = base / 'train.csv'
    csv_val = base / 'val.csv'

    if arrow_train.exists() and arrow_val.exists():
        import pyarrow.ipc as ipc
        print(f'Reading Arrow files from {base}')
        with open(arrow_train, 'rb') as f:
            train_df = ipc.open_stream(f).read_all().to_pandas()
        with open(arrow_val, 'rb') as f:
            val_df = ipc.open_stream(f).read_all().to_pandas()
        return train_df, val_df

    if csv_train.exists() and csv_val.exists():
        print(f'Reading CSV files from {base}')
        train_df = pd.read_csv(csv_train)
        val_df = pd.read_csv(csv_val)
        return train_df, val_df

    return None, None

train_df = None
val_df = None
for candidate in colab_data_candidates:
    if Path(candidate).exists():
        train_df, val_df = load_arrow_or_csv(candidate)
        if train_df is not None:
            break

if train_df is None or val_df is None:
    raise FileNotFoundError(
        'Could not find RACE data. Upload the dataset to /content/race-dataset or mount Google Drive with the files.'
    )

print(f"✓ Loaded {len(train_df)} training samples")
print(f"✓ Loaded {len(val_df)} validation samples\n")

# Display sample
print("Sample data:")
print(train_df.head(1)[['article', 'question', 'options', 'answer']].to_string())

## 6. Load RACE Data in Colab

In [ ]:
def convert_answer_to_label(answer_str):
    """Convert answer letter to index: A->0, B->1, C->2, D->3"""
    answer_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    return answer_map.get(answer_str.strip().upper(), 0)

print("✓ Answer label converter defined")

## 5. Convert Answer Labels

In [ ]:
def build_multiclass_feature_matrix(questions, all_options_list, passages, feature_engineer):
    """
    Build combined feature matrix for multi-class classification.
    
    For each question:
    - Extract features for each of the 4 options
    - Concatenate all features into one vector
    - This allows model to compare across options
    
    Returns X: [N, feature_dim] where feature_dim = 5000*4 + 4*4
    """
    all_features = []
    
    for question, options, passage in zip(questions, all_options_list, passages):
        # One-Hot features for each option
        onehot_parts = []
        for option in options:
            combined = question + ' ' + option
            onehot_feat = feature_engineer.transform_onehot([combined])
            onehot_parts.append(onehot_feat)
        
        # Stack one-hot features from all 4 options
        onehot_all = hstack(onehot_parts)
        
        # Lexical features for all options
        lexical_all = feature_engineer.extract_lexical_features_for_options(
            question, options, passage
        )
        lexical_all_sparse = csr_matrix(lexical_all.flatten()).reshape(1, -1)
        
        # Combine into a single row for this question
        combined_features = hstack([onehot_all, lexical_all_sparse])
        all_features.append(combined_features)
    
    # Stack all question rows vertically
    X = vstack(all_features)
    return X.toarray()

print("✓ Feature matrix builder defined")

## 4. Build the Multi-Class Feature Matrix

In [ ]:
class FeatureEngineer:
    """Feature engineering for multi-class Q&A (selecting correct option)."""
    
    def __init__(self, max_features=5000):
        self.max_features = max_features
        self.onehot_vectorizer = None
        self.scaler = StandardScaler()
    
    def compute_word_overlap(self, text1, text2):
        """Compute word overlap between texts."""
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())
        if len(words1) == 0 or len(words2) == 0:
            return 0.0
        overlap = len(words1 & words2)
        return overlap / max(len(words1), len(words2))
    
    def compute_char_match_score(self, text1, text2):
        """Compute character-level similarity."""
        i = 0
        while i < len(text1) and i < len(text2) and text1[i] == text2[i]:
            i += 1
        return i / max(len(text1), len(text2), 1)
    
    def compute_passage_frequency(self, word, passage):
        """Compute how frequently option appears in passage."""
        words = passage.lower().split()
        if len(words) == 0:
            return 0.0
        return words.count(word.lower()) / len(words)
    
    def extract_lexical_features_for_options(self, question, options, passage):
        """Extract lexical features for all options. Returns: [4, 4] array."""
        features_list = []
        for option in options:
            word_overlap = self.compute_word_overlap(question, option)
            char_match = self.compute_char_match_score(question, option)
            option_length = len(option.split())
            passage_freq = self.compute_passage_frequency(option, passage)
            features = [word_overlap, char_match, option_length, passage_freq]
            features_list.append(features)
        return np.array(features_list)
    
    def fit_onehot(self, texts):
        """Fit One-Hot vectorizer on texts."""
        self.onehot_vectorizer = CountVectorizer(
            max_features=self.max_features,
            lowercase=True,
            binary=True,
            stop_words='english'
        )
        self.onehot_vectorizer.fit(texts)
        return self
    
    def transform_onehot(self, texts):
        """Transform texts using fitted One-Hot vectorizer."""
        if self.onehot_vectorizer is None:
            raise ValueError("One-Hot vectorizer not fitted.")
        return self.onehot_vectorizer.transform(texts)
    
    def save(self, path):
        """Save fitted vectorizers."""
        joblib.dump({
            'onehot': self.onehot_vectorizer,
            'scaler': self.scaler,
            'max_features': self.max_features
        }, path)
    
    @staticmethod
    def load(path):
        """Load saved vectorizers."""
        data = joblib.load(path)
        fe = FeatureEngineer(max_features=data['max_features'])
        fe.onehot_vectorizer = data['onehot']
        fe.scaler = data['scaler']
        return fe

print("✓ FeatureEngineer class defined")

## 3. Define the Feature Engineering Class

In [ ]:
try:
    nltk.data.find('tokenizers/punkt')
    print("✓ NLTK punkt tokenizer already available")
except LookupError:
    print("Downloading NLTK punkt tokenizer...")
    nltk.download('punkt')
    print("✓ NLTK punkt tokenizer downloaded")

## 2. Download and Verify NLTK Resources

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.semi_supervised import LabelPropagation
from sklearn.mixture import GaussianMixture
import xgboost as xgb
from scipy.sparse import csr_matrix, hstack, vstack
import joblib
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except Exception as e:
    print(f"Google Drive mount skipped: {e}")
    print("You can still run the notebook and save to /content/models/")

## 1. Import Libraries and Configure Warnings

# Multi-Class Model A Training for RACE Dataset

This notebook trains Model A to predict which option (0, 1, 2, or 3) is the correct answer for each question in the RACE dataset using multi-class classification.

**Expected Results:**
- Accuracy: 60-80%+ on validation set
- Training time: ~12-18 minutes on GPU
- Output: trained models + feature engineer saved to `/content/models/model_a/traditional/` by default

**Setup:** Make sure GPU is enabled in Colab (Runtime → Change runtime type → GPU)